In [74]:
import pandas as pd
import plotly_express as px
from utils import *

In [75]:
data = last_full_pull()

Loading most recent full pull file...
Loading file: ../Data/Full Pulls\ygo_full_daily_2025-09-21_20-35-33.csv


In [76]:
data.head(5)

,card_id,name,desc,pend_desc,monster_desc,type,subtype,frame,race,archetype,...,min_price,price_range,price_stddev,is_tcg_exclusive,is_ocg_exclusive,days_since_tcg_release,days_since_ocg_release,first_market,market_delay,desc_length
0,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.12,0.00,NaN,0,0,52.0,183.0,OCG,131.0,92
1,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.12,0.00,NaN,0,0,52.0,183.0,OCG,131.0,92
2,34541863,"""A"" Cell Breeding Device","During each of your Standby Phases, put 1 A-Co...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,0.12,24.33,12.004450,0,0,6703.0,6793.0,OCG,90.0,16
3,64163367,"""A"" Cell Incubator",Each time an A-Counter(s) is removed from play...,NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,0.25,1.00,0.473242,0,0,6521.0,6637.0,OCG,116.0,32
4,91231901,"""A"" Cell Recombination Device",Target 1 face-up monster on the field; send 1 ...,NaN,NaN,Spell Card,Quick-Play Spell,spell,Quick-Play,Alien,...,0.22,0.77,0.320728,0,0,3244.0,3361.0,OCG,117.0,66


In [77]:
print("Number of unique *printings*:")
print(data.shape[0])
print("Number of unique *cards*:")
print(data['card_id'].nunique())

Number of unique *printings*:
39781
Number of unique *cards*:
13477


In [78]:
# Write code to filter data to the oldest tcg_date for each card_id
data_all = data.copy()
data_all['tcg_date_dt'] = pd.to_datetime(data_all['tcg_date'], errors='coerce')
oldest_dates = data_all.groupby('card_id')['tcg_date_dt'].min().reset_index()
oldest_dates.columns = ['card_id', 'oldest_tcg_date']
data_all = data_all.merge(oldest_dates, on='card_id')
data = data_all[data_all['tcg_date_dt'] == data_all['oldest_tcg_date']]
data = data.drop(columns=['oldest_tcg_date'])
# Now for any card_id with multiple entries (e.g. multiple printings on the same day), keep only the first one
data = data.drop_duplicates(subset=['card_id'], keep='first')
data.shape

(13476, 57)

In [79]:
# Find the one card_id in data that's not in data_all
set(data['card_id']) - set(data_all['card_id'])

set()

In [80]:
FRAME_COLORS = {
    "Spell Card": "#228B22",                    # Forest Green

    # Monster categories grouped by type similarity and common frame colors
    "Normal Monster": "#EBDF9E",                # Beige/cream - Normal monsters
    "Effect Monster": "#D2691E",                # Brown-orange
    "Flip Effect Monster": "#D97A25",           # Slightly lighter brown-orange, related to Effect Monster
    "Union Effect Monster": "#C2691E",          # Related brown-orange shade
    "Pendulum Effect Monster": "#CC7A32",       # Related to Effect Monster, with distinct slightly lighter tone
    "Pendulum Effect Fusion Monster": "#803080", # Purplish, fusion related
    "Pendulum Normal Monster": "#E9DCC9",       # Very light beige, near normal
    "Synchro Monster": "#FFFFFF",                # White
    "Synchro Tuner Monster": "#E6E6E6",          # Near white, related to Synchro Monster
    "Synchro Pendulum Effect Monster": "#F0F0F0", # Slightly off white, pendulum synchro
    "Tuner Monster": "#CCCCCC",                   # Gray, related to Synchro Tuner
    "Normal Tuner Monster": "#DDDDDD",            # Light gray, normal tuner
    "Gemini Monster": "#D0C5A2",                   # Slightly darker beige, related to Normal/Effect
    "Spirit Monster": "#D0B983",                   # Light goldish, related but distinct
    "Ritual Effect Monster": "#4169E1",            # Royal blue, ritual
    "Ritual Monster": "#3A5FCD",                   # Slightly darker blue, related ritual
    "Toon Monster": "#FFB6C1",                      # Light Pink, cartoonish theme

    # Trap Card
    "Trap Card": "#C71585",                      # Medium Violet Red (pinkish purple)

    # Other monster special frames
    "Link Monster": "#1E90FF",                    # Dodger Blue
    "XYZ Monster": "#000000",                     # Black
    "XYZ Pendulum Effect Monster": "#222222",     # Dark gray, similar to XYZ but distinct
    "Pendulum Effect Ritual Monster": "#3C50B4", # Related to Ritual, pendulum effect
    "Pendulum Flip Effect Monster": "#DB8B3A",    # Orangish, related to pendulum effect
    "Flip Tuner Effect Monster": "#B97416",       # Brownish, closer to tuner effect

    # Less common or specialized cards
    "Skill Card": "#6B8E23",                      # Olive green, distinct from spell cards
    "Token": "#808080",                           # Gray
}


In [81]:
type_counts = data['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']
fig = px.bar(
    type_counts, 
    x='count',
    y='type', 
    category_orders={'type': type_counts.sort_values('count', ascending=False)['type'].tolist()},
    title="Card Type Distribution",
    text='count',
    color='type',
    color_discrete_map=FRAME_COLORS,
    height=800,
    template='ggplot2'
)
fig.update_yaxes(showgrid=False)
fig.show()

In [82]:
data['tcg_date_dt'] = pd.to_datetime(data['tcg_date'], errors='coerce')
data['release_year'] = data['tcg_date_dt'].dt.year
yearly_counts = data[data['release_year'] < 2025].groupby(['type', 'release_year']).size().reset_index(name='count')
fig = px.line(
    yearly_counts,
    x='release_year',
    y='count',
    color='type',
    color_discrete_map=FRAME_COLORS,
    title='Yearly Card Releases by Type',
    markers=True,
    height=850,
    template='ggplot2'
)
fig.update_xaxes(showgrid=False)
fig.show()